# 🎲 Advanced AI Gamemaster - OpenEnv RL Training with Unsloth

This notebook implements **Multi-Dimensional Reward GRPO** to train a self-improving Gamemaster.

### Advanced RL Strategies Implemented:
1. **Weighted Reward Shaping:** We prioritize Rule Accuracy (60%) while still rewarding Narrative Consistency (20%) and Progression (20%).
2. **Group Relative Policy Optimization:** We sample 8 completions per turn to allow the model to compare successful vs. unsuccessful rule enforcement.
3. **OpenEnv State Persistence:** The agent is trained against a live Infinite Dungeon engine.

In [ ]:
# 1. SETUP: Clone your project from GitHub to Colab
!git clone https://github.com/http-pruthvi/gamemaster.git
import os
os.chdir('gamemaster/gamemaster_env')

print("Current working directory:", os.getcwd())

# 2. INSTALL: Dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install openenv-core pydantic

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import asyncio
import re
import json
from client import GamemasterEnv
from models import GamemasterAction

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)

### Multi-Dimensional Reward Functions
We define separate functions for each dimension. TRL will average these.

In [ ]:
ENV_URL = "https://Pruthvi1762-gamemaster-env.hf.space/api"

def get_env_result(generated_text):
    try:
        json_match = re.search(r'\{.*\}', generated_text, re.DOTALL)
        if not json_match: return None
        action_data = json.loads(json_match.group(0))
        action = GamemasterAction(**action_data)
        
        with GamemasterEnv(base_url=ENV_URL).sync() as client:
            client.reset()
            return client.step(action)
    except: return None

def rule_accuracy_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("rule_accuracy", -1.0) 
            if get_env_result(c[0]["content"]) else -2.0 for c in completions]

def progression_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("progression", 0.0) 
            if get_env_result(c[0]["content"]) else 0.0 for c in completions]

def narrative_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("narrative_quality", 0.0) 
            if get_env_result(c[0]["content"]) else 0.0 for c in completions]

In [ ]:
SYSTEM_PROMPT = """
You are an AI Gamemaster. You must enforce the rules of the game while telling a good story.
You will receive an Observation containing the player's action and the system's dice roll (1-20).
If the player attacks, a roll of >= 10 is a hit. You must apply damage.
If the roll is < 10, it is a miss. Do not apply damage.

You MUST respond ONLY with a valid JSON object matching this schema:
{
  "narrative_response": "Your story text here",
  "target_to_damage": "goblin" or null,
  "damage_amount": integer (0 if miss, >0 if hit),
  "item_to_give": "item name" or null
}
"""

scenarios = [
    {"observation": "Player: 'I attack the goblin!' | Dice Roll: 15 | Goblin HP: 10"},
    {"observation": "Player: 'I swing my sword at the goblin.' | Dice Roll: 4 | Goblin HP: 10"},
    {"observation": "Player: 'I search the dead goblin for loot.' | Dice Roll: 12 | Goblin HP: 0"},
    {"observation": "Player: 'I attack the goblin!' | Dice Roll: 20 | Goblin HP: 10"},
]

dataset_dict = {"prompt": []}
for s in scenarios * 25: 
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": s["observation"]}
    ]
    dataset_dict["prompt"].append(prompt)

train_dataset = Dataset.from_dict(dataset_dict)

In [ ]:
training_args = GRPOConfig(
    learning_rate = 5e-6,
    num_generations = 8, 
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    max_completion_length = 512,
    logging_steps = 1,
    max_prompt_length = 512,
    num_train_epochs = 1,
    output_dir = "outputs",
    report_to = "wandb", # HACKATHON RULE: Experimental Tracking
    run_name = "gamemaster-rl-grpo",
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [
        rule_accuracy_reward,
        progression_reward,
        narrative_reward
    ],
    args = training_args,
    train_dataset = train_dataset,
)

trainer.train()